In this notebook we'll experiment with different parameters for the RecursiveCharacterTextSplitter splitter from LangChain, and decide which are most suitable for our documents.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

from rag_pipeline.parse_documents import parse_document

In [ ]:
# doc1 contains 4 columns per page, so I'm testing whether the loader can handle this layout

doc1_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/motion_sickness_PIL.pdf"

# doc 2 is a more standard layout, with 1 column per page

doc2_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/antacid_PIL.pdf"

In [ ]:
clean_text = parse_document(doc1_path)

page_1_clean_text = clean_text[0].page_content
print(page_1_clean_text)

In [ ]:
page_1_chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150
).split_text(page_1_clean_text)

In order to set `chunk_size`, we need to take into account the maximum sequence length of the embedding model that we'll be using. We can revisit this once we've chosen an embedding model, but for now let's work off of the assumption that 256 is a safe number of tokens for most sentence-transformer models. Assuming ~4 characters per token, this gives a chunk size of ~1000.

Note that in our production-ready code, parameters such as `chunk_size` will be set in a config file, not hardcoded.

It is generally recommended that the overlap between chunks should be set to 10-20% of the chunk size. We'll take 15% of 1000 which is 150. It's good to remember that the purpose of this overlap is to maintain context between one chunk and the next. Without overlap, an idea could get split between two chunks. However, setting the overlap too high wastes storage space in the vector store and wastes compute during retrieval.

`RecursiveCharacterTextSplitter` also has a `separators` parameter which defaults to `["\n\n", "\n", " ", ""]`. This means it first tries to split on paragraphs, then new lines, then words, then characters.

In [ ]:
n = 1
print(page_1_chunks[n])

LangChain splitters also have a `split_documents` attribute. The resulting chunks are `langchain_core.documents.base.Document` objects, just like the outputs of our `parse_document` function. This way we can persist the metadata from pages to chunks.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

for document in clean_text:
    for chunk_index, chunk in enumerate(splitter.split_documents([document])):
        print(chunk_index, chunk.metadata)